# Health-JEPA — All of Us Extraction Pipeline
## Lightweight Cardiometabolic Cohort (EHR + Wearables, Weekly Bins)

A **memory-friendly** version of the extraction that still produces a rich
`(x, z, y)` tensor for `CausalJEPA`. Four tricks that drop CPU + RAM load
by an order of magnitude versus the original notebook:

1. **Server-side weekly aggregation** — labs and wearables are averaged per
   ISO week in BigQuery, so `T` is bounded at ~`WINDOW_DAYS / 7` (≈104 steps
   at a ±365-day window) instead of thousands of daily rows per patient.
2. **Per-arm cohort cap** (`MAX_PER_ARM`) — random sample each arm so you
   are not always pulling 45k patients just to debug the pipeline.
3. **Focused lab vocabulary** — 14 high-value cardiometabolic labs + 5 vitals
   instead of ~28 / 3667. Smaller `F`, same clinical story.
4. **Automatic checkpoints** — each heavy DataFrame is saved to
   `pipeline_checkpoints/*.parquet` so a kernel crash does not force a full
   re-run; a restart can skip straight to the pivot with `load_checkpoint`.

Pivot is fully vectorised (NumPy fancy indexing, no `iterrows`).

## 1. Setup

In [ ]:
import os
import gc
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

dataset = os.environ["WORKSPACE_CDR"]
print(f"Using CDR dataset: {dataset}")

def run_bq(sql: str) -> pd.DataFrame:
    """Submit a BigQuery job and return a DataFrame."""
    return pd.read_gbq(sql, dialect="standard", progress_bar_type=None)

# --- Checkpoint helpers (persistent disk) ---
CHECKPOINT_DIR = Path("pipeline_checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def save_checkpoint(name: str, df: pd.DataFrame) -> None:
    path = CHECKPOINT_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"[checkpoint] saved {name}: {len(df):,} rows → {path}")

def load_checkpoint(name: str) -> pd.DataFrame:
    path = CHECKPOINT_DIR / f"{name}.parquet"
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_parquet(path)
    print(f"[checkpoint] loaded {name}: {len(df):,} rows ← {path}")
    return df

## 2. Configuration

- **`WINDOW_DAYS`** — half-width of the observation window around the index
  date. Weekly binning means the final tensor has at most `WINDOW_DAYS/7`
  pre- and post-steps. 365 keeps a year on each side at only ~52 steps.
- **`MAX_PER_ARM`** — upper cap on patients per drug. Scale this up once
  the pipeline survives end-to-end. `None` → no cap (every eligible person).
- **`MIN_FEATURES_PRE`** / **`MIN_FEATURES_POST`** — drop patients with too
  little signal; saves memory in the pivot.

In [ ]:
WINDOW_DAYS = 365
MAX_PER_ARM = 2000      # set to None to disable cap
MIN_FEATURES_PRE = 1    # at least N observed feature-cells in pre window
MIN_FEATURES_POST = 1
RANDOM_SEED = 42

INTERVENTION_CONCEPTS: dict[str, int] = {
    "metformin":    1503297,
    "atorvastatin": 1545958,
    "lisinopril":   1308216,
}

concept_to_z    = {v: i for i, v in enumerate(INTERVENTION_CONCEPTS.values())}
concept_id_list = ", ".join(str(v) for v in INTERVENTION_CONCEPTS.values())
z_to_label      = {i: k for i, k in enumerate(INTERVENTION_CONCEPTS.keys())}
id_to_label     = {v: k for k, v in INTERVENTION_CONCEPTS.items()}

print(f"K = {len(INTERVENTION_CONCEPTS)} interventions")
print(f"Intervention → z: {concept_to_z}")
print(f"Window: ±{WINDOW_DAYS} days (binned weekly → T_max ≈ {WINDOW_DAYS//7})")
print(f"Per-arm cap: {MAX_PER_ARM}")

### 2a. Verify ingredients exist in this CDR

In [ ]:
q_verify = f"""
SELECT
  c.concept_id,
  c.concept_name,
  c.standard_concept,
  c.vocabulary_id,
  c.domain_id,
  (SELECT COUNT(DISTINCT ca.descendant_concept_id)
     FROM `{dataset}.concept_ancestor` ca
    WHERE ca.ancestor_concept_id = c.concept_id) AS n_descendants
FROM `{dataset}.concept` c
WHERE c.concept_id IN ({concept_id_list})
ORDER BY c.concept_id
"""
run_bq(q_verify)

## 3. Build the intervention cohort (with per-arm cap)

`FirstExposure` gets the earliest descendant exposure per person × ingredient.
`Capped` randomly samples up to `MAX_PER_ARM` per arm so memory stays sane.

In [ ]:
_cap_clause = (
    f"WHERE rn <= {MAX_PER_ARM}" if MAX_PER_ARM is not None else ""
)

COHORT_CTE = f"""
WITH Descendants AS (
  SELECT ca.ancestor_concept_id  AS intervention_id,
         ca.descendant_concept_id
  FROM `{dataset}.concept_ancestor` ca
  WHERE ca.ancestor_concept_id IN ({concept_id_list})
),
FirstExposure AS (
  SELECT de.person_id,
         d.intervention_id,
         MIN(de.drug_exposure_start_date) AS index_date
  FROM `{dataset}.drug_exposure` de
  JOIN Descendants d ON de.drug_concept_id = d.descendant_concept_id
  GROUP BY de.person_id, d.intervention_id
),
Ranked AS (
  SELECT person_id, intervention_id, index_date,
         ROW_NUMBER() OVER (
           PARTITION BY intervention_id
           ORDER BY FARM_FINGERPRINT(CAST(person_id AS STRING) || '|{RANDOM_SEED}')
         ) AS rn
  FROM FirstExposure
),
Cohort AS (
  SELECT person_id, intervention_id, index_date
  FROM Ranked
  {_cap_clause}
)
"""

print("Building cohort …")
df_cohort = run_bq(COHORT_CTE + "SELECT * FROM Cohort")
save_checkpoint("df_cohort", df_cohort)

print(f"\nCohort size: {len(df_cohort):,} (person × intervention) pairs "
      f"across {df_cohort['person_id'].nunique():,} unique participants.")
print()
print(df_cohort["intervention_id"].map(id_to_label).value_counts())

## 4. Focused feature vocabulary

~20 features instead of thousands. All are **standard LOINC** concept_ids
well-represented in All of Us.

In [ ]:
LAB_FEATURES: dict[str, int] = {
    # --- Core cardiometabolic labs ---
    "hba1c":         3004410,
    "glucose":       3004501,
    "ldl_chol":      3028437,
    "hdl_chol":      3007070,
    "total_chol":    3027114,
    "triglycerides": 3022192,
    # --- Renal ---
    "creatinine":    3016723,
    "egfr":          3049187,
    # --- Hematology / chemistry (quick health signal) ---
    "hemoglobin":    3000963,
    "wbc":           3000905,
    "potassium":     3023103,
    "sodium":        3019550,
    "alt":           3006923,
    "albumin":       3024561,
    # --- Vitals ---
    "systolic_bp":   3004249,
    "diastolic_bp":  3012888,
    "bmi":           3038553,
    "resting_hr":    3027018,
    "spo2":          3016502,
}
lab_id_to_name = {v: k for k, v in LAB_FEATURES.items()}
ehr_concept_list = ", ".join(str(v) for v in LAB_FEATURES.values())

print(f"Curated EHR features: {len(LAB_FEATURES)}")

## 5. Labs & vitals — **weekly** aggregation in BigQuery

Averaging per ISO-week in SQL means we **download dozens of rows per patient**
instead of dozens per day. Typical shrink: 10–100× fewer rows to parse in
pandas. Numeric outliers are filtered via a basic value range (drops obvious
unit / data-entry errors).

In [ ]:
q_labs = COHORT_CTE + f"""
SELECT
  c.person_id,
  c.intervention_id,
  m.measurement_concept_id AS feature_id,
  CAST(FLOOR(DATE_DIFF(m.measurement_date, c.index_date, DAY) / 7.0) AS INT64)
    AS week_from_index,
  AVG(m.value_as_number) AS value,
  COUNT(*) AS n_obs
FROM Cohort c
JOIN `{dataset}.measurement` m
  ON m.person_id = c.person_id
WHERE m.measurement_concept_id IN ({ehr_concept_list})
  AND m.value_as_number IS NOT NULL
  AND ABS(m.value_as_number) < 1e6
  AND m.measurement_date BETWEEN DATE_SUB(c.index_date, INTERVAL {WINDOW_DAYS} DAY)
                             AND DATE_ADD(c.index_date, INTERVAL {WINDOW_DAYS} DAY)
GROUP BY 1, 2, 3, 4
"""

print("Extracting weekly-aggregated labs …")
df_labs = run_bq(q_labs)
df_labs["feature_key"] = df_labs["feature_id"].map(lab_id_to_name).astype("string")
df_labs = df_labs.dropna(subset=["feature_key"])
save_checkpoint("df_labs", df_labs)

print(f"\nLab rows (weekly): {len(df_labs):,} across "
      f"{df_labs['person_id'].nunique():,} patients")
print()
print(df_labs["feature_key"].value_counts())

## 6. Wearables — weekly aggregation per participant

Each Fitbit table is rolled up to **mean per ISO week**. Participants without
a device just have no wearable rows; the encoder's mask handles missingness.

In [ ]:
def _wear_sql(select_cols: str, table: str, date_col: str, extra_where: str = "") -> str:
    return COHORT_CTE + f"""
    SELECT
      c.person_id,
      c.intervention_id,
      CAST(FLOOR(DATE_DIFF(a.{date_col}, c.index_date, DAY) / 7.0) AS INT64)
        AS week_from_index,
      {select_cols}
    FROM Cohort c
    JOIN `{dataset}.{table}` a ON a.person_id = c.person_id
    WHERE a.{date_col} BETWEEN DATE_SUB(c.index_date, INTERVAL {WINDOW_DAYS} DAY)
                           AND DATE_ADD(c.index_date, INTERVAL {WINDOW_DAYS} DAY)
      {extra_where}
    GROUP BY 1, 2, 3
    """

def _extract_activity() -> pd.DataFrame:
    sel = """
      AVG(CAST(a.steps                  AS FLOAT64)) AS wear_steps,
      AVG(CAST(a.sedentary_minutes      AS FLOAT64)) AS wear_sedentary_min,
      AVG(CAST(a.lightly_active_minutes AS FLOAT64)) AS wear_light_active_min,
      AVG(CAST(a.fairly_active_minutes  AS FLOAT64)
        + CAST(a.very_active_minutes    AS FLOAT64)) AS wear_mod_vig_active_min
    """
    df = run_bq(_wear_sql(sel, "activity_summary", "date"))
    return df.melt(
        id_vars=["person_id", "intervention_id", "week_from_index"],
        var_name="feature_key", value_name="value",
    )

def _extract_heart_rate() -> pd.DataFrame:
    sel = """
      AVG(CAST(a.min_heart_rate AS FLOAT64)) AS wear_hr_min,
      AVG(CAST(a.max_heart_rate AS FLOAT64)) AS wear_hr_max
    """
    df = run_bq(_wear_sql(sel, "heart_rate_summary", "date"))
    return df.melt(
        id_vars=["person_id", "intervention_id", "week_from_index"],
        var_name="feature_key", value_name="value",
    )

def _extract_sleep() -> pd.DataFrame:
    sel = """
      AVG(CAST(a.minute_asleep AS FLOAT64)) AS wear_sleep_asleep_min,
      AVG(CAST(a.minute_deep   AS FLOAT64)) AS wear_sleep_deep_min,
      AVG(CAST(a.minute_rem    AS FLOAT64)) AS wear_sleep_rem_min
    """
    extra = "AND (a.is_main_sleep IS NULL OR a.is_main_sleep = TRUE)"
    df = run_bq(_wear_sql(sel, "sleep_daily_summary", "sleep_date", extra))
    return df.melt(
        id_vars=["person_id", "intervention_id", "week_from_index"],
        var_name="feature_key", value_name="value",
    )

wear_frames: list[pd.DataFrame] = []
for name, fn in [
    ("activity_summary",    _extract_activity),
    ("heart_rate_summary",  _extract_heart_rate),
    ("sleep_daily_summary", _extract_sleep),
]:
    try:
        print(f"Extracting {name} (weekly) …")
        w = fn().dropna(subset=["value"])
        print(f"  {len(w):,} rows, {w['person_id'].nunique():,} participants")
        wear_frames.append(w)
    except Exception as e:
        print(f"  [{name}] skipped: {type(e).__name__}: {str(e)[:180]}")

if wear_frames:
    df_wear = pd.concat(wear_frames, ignore_index=True)
else:
    df_wear = pd.DataFrame(
        columns=["person_id", "intervention_id",
                 "week_from_index", "feature_key", "value"]
    )

save_checkpoint("df_wear", df_wear)
print(f"\nTotal wearable rows: {len(df_wear):,}")
if len(df_wear):
    print(df_wear["feature_key"].value_counts())

## 7. Unify EHR + wearables

Both frames now share a `week_from_index` column; we stack them as one long
table keyed on `(person_id, intervention_id, week_from_index, feature_key)`.

In [ ]:
lab_slim = df_labs[[
    "person_id", "intervention_id", "week_from_index", "feature_key", "value",
]].copy()

if len(df_wear):
    df_raw = pd.concat([lab_slim, df_wear], ignore_index=True)
else:
    df_raw = lab_slim

df_raw["week_from_index"] = df_raw["week_from_index"].astype(np.int32)
df_raw["value"]           = df_raw["value"].astype(np.float32)
df_raw["feature_key"]     = df_raw["feature_key"].astype("string")

# Free the intermediates – big win on small VMs
del lab_slim, df_labs, df_wear
gc.collect()

save_checkpoint("df_raw", df_raw)
print(f"\ndf_raw: {len(df_raw):,} rows, "
      f"{df_raw['person_id'].nunique():,} patients, "
      f"{df_raw['feature_key'].nunique()} distinct features")
print()
print("Rows per arm:")
print(df_raw["intervention_id"].map(id_to_label).value_counts())

## 8. Feature vocabulary (EHR first, wearables after)

Fixed ordering → deterministic column indices shared between training and
inference.

In [ ]:
present_keys = set(df_raw["feature_key"].dropna().unique())
lab_keys  = [k for k in LAB_FEATURES.keys() if k in present_keys]
wear_keys = sorted(str(k) for k in present_keys if str(k).startswith("wear_"))

all_keys = lab_keys + wear_keys
feature_to_col: dict[str, int] = {k: i for i, k in enumerate(all_keys)}
num_features = len(feature_to_col)

df_raw["col"] = df_raw["feature_key"].map(feature_to_col).astype("Int32")
df_raw = df_raw.dropna(subset=["col"])
df_raw["col"] = df_raw["col"].astype(np.int32)

print(f"Feature vocabulary size F = {num_features}")
print(f"  EHR features:      {len(lab_keys)} → {lab_keys}")
print(f"  Wearable features: {len(wear_keys)} → {wear_keys}")

## 9. Pivot to per-patient `(x, z, y)` tensors (vectorised)

Timestamps are stored in **seconds since day 0** so the existing JEPA
`ContinuousTemporalEncoding` works unchanged (1 week = 604 800 s).

In [ ]:
SECONDS_PER_DAY = 86_400
SECONDS_PER_WEEK = 7 * SECONDS_PER_DAY

def _window_to_arrays(window_df: pd.DataFrame, F: int):
    if window_df.empty:
        return None
    weeks = window_df["week_from_index"].to_numpy()
    cols  = window_df["col"].to_numpy()
    vals  = window_df["value"].to_numpy(dtype=np.float32)

    unique_weeks, t_idx = np.unique(weeks, return_inverse=True)
    T = len(unique_weeks)

    values = np.zeros((T, F), dtype=np.float32)
    mask   = np.zeros((T, F), dtype=np.float32)
    values[t_idx, cols] = vals
    mask[t_idx, cols]   = 1.0
    timestamps = unique_weeks.astype(np.float64) * SECONDS_PER_WEEK
    return values, mask, timestamps

patients: list[dict] = []
skipped_empty = 0
skipped_sparse = 0
F = num_features

df_raw = df_raw.sort_values(["person_id", "intervention_id", "week_from_index"])

for (person_id, intervention_id), g in df_raw.groupby(
    ["person_id", "intervention_id"], sort=False
):
    pre  = g[g["week_from_index"] < 0]
    post = g[g["week_from_index"] > 0]

    pre_arr  = _window_to_arrays(pre,  F)
    post_arr = _window_to_arrays(post, F)
    if pre_arr is None or post_arr is None:
        skipped_empty += 1
        continue

    pre_v, pre_m, pre_t = pre_arr
    post_v, post_m, post_t = post_arr
    if pre_m.sum() < MIN_FEATURES_PRE or post_m.sum() < MIN_FEATURES_POST:
        skipped_sparse += 1
        continue

    patients.append({
        "person_id":       int(person_id),
        "intervention_z":  concept_to_z[int(intervention_id)],
        "pre_values":      pre_v,
        "pre_mask":        pre_m,
        "pre_timestamps":  pre_t,
        "post_values":     post_v,
        "post_mask":       post_m,
        "post_timestamps": post_t,
    })

del df_raw
gc.collect()

print(f"Formatted {len(patients):,} valid patient-intervention pairs.")
print(f"Skipped   {skipped_empty:,} missing pre or post window.")
print(f"Skipped   {skipped_sparse:,} too sparse.")

## 10. Sanity checks

In [ ]:
if patients:
    s = patients[0]
    print(f"Example patient (z={s['intervention_z']} = {z_to_label[s['intervention_z']]}):")
    print(f"  pre  values {s['pre_values'].shape}, mask {s['pre_mask'].shape}, ts {s['pre_timestamps'].shape}")
    print(f"  post values {s['post_values'].shape}, mask {s['post_mask'].shape}, ts {s['post_timestamps'].shape}")
    print(f"  pre  observation density:  {s['pre_mask'].mean():.2%}")
    print(f"  post observation density: {s['post_mask'].mean():.2%}")

z_counts = pd.Series([p["intervention_z"] for p in patients]).value_counts().sort_index()
print("\nFinal per-arm training counts:")
for z_idx, count in z_counts.items():
    print(f"  z={int(z_idx)} ({z_to_label[int(z_idx)]}): {count:,} patients")

## 11. Export training shards

In [ ]:
OUTPUT_DIR = Path("training_data")
OUTPUT_DIR.mkdir(exist_ok=True)

vocab_path = OUTPUT_DIR / "feature_vocab.json"
with open(vocab_path, "w") as f:
    json.dump(feature_to_col, f, indent=2)

interventions_path = OUTPUT_DIR / "intervention_map.json"
with open(interventions_path, "w") as f:
    json.dump({
        "concept_to_z": {str(k): v for k, v in concept_to_z.items()},
        "labels":       {str(v): k for k, v in INTERVENTION_CONCEPTS.items()},
    }, f, indent=2)

npz_path = OUTPUT_DIR / "patient_tensors.npz"
np.savez_compressed(
    npz_path,
    person_ids=np.array([p["person_id"] for p in patients]),
    intervention_z=np.array([p["intervention_z"] for p in patients]),
    pre_values=np.array([p["pre_values"] for p in patients], dtype=object),
    pre_mask=np.array([p["pre_mask"] for p in patients], dtype=object),
    pre_timestamps=np.array([p["pre_timestamps"] for p in patients], dtype=object),
    post_values=np.array([p["post_values"] for p in patients], dtype=object),
    post_mask=np.array([p["post_mask"] for p in patients], dtype=object),
    post_timestamps=np.array([p["post_timestamps"] for p in patients], dtype=object),
)

manifest = {
    "cohort_id":             "aou_cardiometabolic_weekly_v2",
    "focus":                 "cardiometabolic (metformin / atorvastatin / lisinopril)",
    "index_definition":      "First descendant drug_exposure per person × ingredient",
    "pre_window_days":       WINDOW_DAYS,
    "post_window_days":      WINDOW_DAYS,
    "time_resolution":       "weekly",
    "max_per_arm":           MAX_PER_ARM,
    "num_interventions":     len(INTERVENTION_CONCEPTS),
    "num_features":          num_features,
    "num_ehr_features":      len(lab_keys),
    "num_wearable_features": len(wear_keys),
    "num_patients":          len(patients),
    "feature_vocab_path":    str(vocab_path),
    "intervention_map_path": str(interventions_path),
    "tensor_path":           str(npz_path),
    "cdr_dataset":           dataset,
}
manifest_path = OUTPUT_DIR / "manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("Manifest:")
for k, v in manifest.items():
    print(f"  {k}: {v}")

---

### After a crash (quick recovery)

Restart the kernel, then run only cells **1 (Setup)**, **2 (Configuration)**
and **4 (Feature vocabulary)**. Skip BigQuery entirely with:

```python
df_raw = load_checkpoint("df_raw")
```

Then jump straight to **Step 8 → 9 → 10 → 11**.